In [ ]:
import pandas as pd
from pathlib import Path
import json
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.preprocessing import MinMaxScaler

In [ ]:
DATA_PATH = '../preprocessed_data/'

In [ ]:
preprocessed_data = []

In [ ]:
for file in Path(DATA_PATH).glob('*.json'):
    file = json.load(open(file))
    for item in file:  
        preprocessed_data.append(item)

In [ ]:
df = pd.DataFrame(preprocessed_data)
print(df.head())

In [ ]:
df_flat = df.copy()

print(df_flat['tables'])

# Để đếm tần suất xuất hiện của cột trong query:
df_flat['num_tables'] = df_flat['tables'].apply(len)
df_flat['has_join'] = df_flat['join_columns'].apply(lambda x: 1 if len(x) > 0 else 0)
df_flat['is_aggregate'] = df_flat['columns_group'].apply(lambda x: 1 if len(x) > 0 else 0)

print(f"Tổng số truy vấn đã gộp: {len(df_flat)}")
print(df_flat.columns)

In [ ]:
class ReflectionBrain:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        # Sử dụng mô hình NLP nhẹ, phù hợp với phần cứng (hardware-friendly)
        self.encoder = SentenceTransformer(model_name)
        self.knowledge_base = {} # Lưu trữ: {col_name: {'dim_score': 0, 'mea_score': 0, 'embedding': vector}}
        self.threshold = 0.7 # Ngưỡng tương đồng để so sánh cột mới

    def _extract_list(self, value):
        """Chuyển đổi các chuỗi list trong DataFrame thành list thực tế"""
        if isinstance(value, str):
            return [x.strip() for x in value.replace('[','').replace(']','').replace("'", "").split(',') if x]
        return value if isinstance(value, list) else []

import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

class ReflectionBrain:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.encoder = SentenceTransformer(model_name)
        self.knowledge_base = {} 
        self.threshold = 0.7 

    def _extract_list(self, value):
        if isinstance(value, str):
            return [x.strip() for x in value.replace('[','').replace(']','').replace("'", "").split(',') if x]
        return value if isinstance(value, list) else []

    # --- PHẦN HOÀN THIỆN: HỌC TỪ DỮ LIỆU ĐÃ DÁN NHÃN ---
    def train_from_labeled_data(self, labeled_list):
        """
        Học từ danh sách các chuỗi JSON đã được dán nhãn (labeled data).
        Dữ liệu đầu vào: ["{\"column\": \"PLAYER_ID\", \"type\": \"dimension\"}", ...]
        """
        for item in labeled_list:
            try:
                # 1. Parse chuỗi JSON thành dictionary
                data = json.loads(item)
                col_name = data.get('column')
                label = data.get('type', '').lower()

                if not col_name or not label:
                    continue

                # 2. Map nhãn sang định dạng dim/mea của knowledge_base
                label_type = 'dim' if 'dimension' in label else 'mea'
                
                # 3. Cập nhật vào kho tri thức
                self._update_knowledge(col_name, label_type)
                
            except json.JSONDecodeError:
                print(f"Lỗi: Không thể parse chuỗi JSON: {item}")
                continue

        # 4. Re-build lại embeddings sau khi cập nhật dữ liệu mới
        self._build_embeddings()

    def train(self, df):
        """
        Bước 2: Context Mapping - Học từ lịch sử truy vấn
        """
        for _, row in df.iterrows():
            # Lấy danh sách cột từ các feature quan trọng
            groups = self._extract_list(row['columns_group'])
            wheres = self._extract_list(row['columns_where'])
            selects = self._extract_list(row['columns_select'])
            is_agg = row['is_aggregate']

            # Logic Context Mapping:
            # 1. Cột xuất hiện trong GROUP BY hoặc WHERE thường là Dimension
            for col in set(groups + wheres):
                self._update_knowledge(col, 'dim')

            # 2. Cột xuất hiện trong SELECT của một câu lệnh Aggregate nhưng KHÔNG nằm trong GROUP BY
            # thường là Measure (ví dụ: SUM(sales) -> sales là measure)
            if is_agg:
                potential_measures = set(selects) - set(groups)
                for col in potential_measures:
                    self._update_knowledge(col, 'mea')

        # Sau khi train, tạo Embedding cho các từ khóa đã học
        self._build_embeddings()

    def _update_knowledge(self, col_name, label_type):
        if col_name not in self.knowledge_base:
            self.knowledge_base[col_name] = {'dim_score': 0, 'mea_score': 0}
        
        if label_type == 'dim':
            self.knowledge_base[col_name]['dim_score'] += 1
        else:
            self.knowledge_base[col_name]['mea_score'] += 1

    def _build_embeddings(self):
        """Bước 1: NLP Embedding - Biến tên cột thành vector"""
        names = list(self.knowledge_base.keys())
        embeddings = self.encoder.encode(names)
        for i, name in enumerate(names):
            self.knowledge_base[name]['embedding'] = embeddings[i]

    def predict_reflection(self, new_table_columns):
        """
        Bước 3: Dự đoán cấu hình cho bảng mới (ví dụ: bảng 'albums')
        """
        suggestions = []
        known_names = list(self.knowledge_base.keys())
        known_embeddings = np.array([v['embedding'] for v in self.knowledge_base.values()])

        # Embedding các cột của bảng mới
        new_embeddings = self.encoder.encode(new_table_columns)

        for i, col in enumerate(new_table_columns):
            # So sánh độ tương đồng cosine giữa cột mới và bộ từ điển đã học
            cos_scores = util.cos_sim(new_embeddings[i], known_embeddings)[0]
            best_match_idx = np.argmax(cos_scores)
            max_score = cos_scores[best_match_idx]

            if max_score >= self.threshold:
                matched_name = known_names[best_match_idx]
                info = self.knowledge_base[matched_name]
                
                # Quyết định dựa trên điểm số đã học
                label = "Dimension" if info['dim_score'] >= info['mea_score'] else "Measure"
                confidence = float(max_score)
            else:
                # Nếu không tìm thấy từ tương đồng, mặc định dựa trên heuristic cơ bản
                label = "Dimension" # Hoặc logic mặc định khác
                confidence = 0.0

            suggestions.append({
                'column': col,
                'suggested_type': label,
                'similarity': confidence,
                'matched_with': matched_name if max_score >= self.threshold else "None"
            })
        
        return pd.DataFrame(suggestions)

In [19]:
with open('../labeled/labeled_columns.json', 'r') as f:
    labeled_list = json.load(f)

with open('../labeled/labeled_columns_02.json', 'r') as f:
    labeled_list.extend(json.load(f))

In [20]:
brain = ReflectionBrain()
brain.train(df_flat)
brain.train_from_labeled_data(labeled_list)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9792.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
test_vector = brain.encoder.encode(["songid"])
print(f"Hình dạng mảng: {test_vector.shape}")# Kết quả nên là 384 cho MiniLM-L6

Hình dạng mảng: (1, 384)


In [22]:
new_cols = ['track_id', 'album_price', 'category']
result = brain.predict_reflection(new_cols)

print("Gợi ý cấu hình Reflection ban đầu:")
print(result)

Gợi ý cấu hình Reflection ban đầu:
        column suggested_type  similarity matched_with
0     track_id      Dimension    1.000000     track_id
1  album_price        Measure    0.718025     album_id
2     category      Dimension    1.000000     category


In [23]:
#save the model
import pickle
with open('reflection_brain.pkl', 'wb') as f:
    pickle.dump(brain, f)